# Julia notebook to compute the symbolic solution to the dissipative buoyancy equation
twnh 

Dec '25   Write `ExampleSolution_v0.5.ipynb` <br>
Jun '26.  Write `BuoyancyEquationSolution.ipynb`

This notebook solves the non-dimensional, linear, time-independent buoyany equation with relaxation:
\begin{align}
Ek \frac{d}{dz} \left( \kappa \frac{d b}{d z} \right) + \Gamma ( B - b ) = 0
\end{align}
for $b(z)$, given diffusivity $\kappa = \kappa (z)$ (customized so $\kappa = 1$ at the moment), and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= b_s ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field, $b_s$ is the specified surface buoyancy field, and $\Gamma$ is the relaxation rate.

In [ ]:
using SymPy

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
# H      = SymFunction("H", real=true, positive=true) # Domain depth H
H      = symbols("H",     real=true, positive=true) # Domain depth
# geometry_params = (H(x,y), z, ξ)
geometry_params = (H, z, ξ)

Ek     = symbols("Ek ",   real=true, positive=true) # Ekman number

# Buoyancy equation symbolic parameters:
b      = SymFunction("b", real=true)                # Buoyancy field b(x,y,z)
ψ      = symbols("ψ",     real=true, positive=true) # Diffusivity parameters
Γ      = symbols("Γ",     real=true, positive=true) # Relaxation parameter
B      = SymFunction("B", real=true)                # Relaxation buoyancy field
bs     = SymFunction("bs",real=true)                # Surface buoyancy field bs(x,y)

# Define diffusivity profile here:
κ      = symbols("κ",     real=true, positive=true) # Diffusivity parameter
Γ_subs =  ψ^2 * Ek * κ

# Define dictionaries for substitutions:
b_params = (κ, Ek , Γ) ;

#### Function to solve the buoyancy equation using a Green's function. This is where `SymPy` is unavoidable as there's no `Symbolics` equivalent to `dsolve` as of July 2026.

In [ ]:
function compute_Gb(b_params, geometry_params)
    # Setup symbols and parameters:
    κexpr, Ek , Γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A", real=true)                                   # Unknown coefficient in the Green's function solution
    
    # 0. Define the ODE for b(z):
    ode = Eq( Ek * diff(κexpr *  diff(b(z),z),z) - Γ * b(z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bm)) == 0                  # Check solution
    @assert sympy.simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    bm = bm.subs(first(bm_const), A)                                    # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bp)) == 0                  # Check solution
    @assert sympy.simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = sympy.simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; ξ)$:
    Gm = sympy.simplify(bm * bp.subs(z,ξ) / (κexpr.subs(z,ξ) * W.subs(z,ξ)))
    Gp = sympy.simplify(bm.subs(z,ξ) * bp / (κexpr.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(κexpr.subs(z,ξ)) == 0

    # Define piecewise Green's function:
    G = sympy.simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert sympy.simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert sympy.simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

### Compute the buoyancy Green's function:

In [ ]:
Gb_sym  = compute_Gb(b_params,geometry_params)
Gb = Gb_sym.subs(Γ,Γ_subs)
println("1.1 Symbolic computation of Gb completed.")
display(Gb)

# Save the expression string to a text file
open("sympy_Gb_expression.txt", "w") do io
    write(io, string(Gb))
end
println("Wrote sympy_Gb_expression.txt file.")

In [ ]:
# Alternative form for Gb in terms of exp(ψξ) and exp(-ψξ):
for piece in Gb.args
    (expr, cond) = piece.args[1], piece.args[2]
    b = expand(expr)
    println("Condition: ", cond)
    collected = collect(b, exp(ψ*ξ))
    plus_term = collected.coeff( exp(ψ*ξ))
    minus_term = collected.coeff( exp(-ψ*ξ))
    println("Coeff exp(+ψξ): ")
    display(simplify(plus_term))
    println()
    println("Coeff exp(-ψξ): ")
    display(simplify(minus_term))
    println()
end
println("1.11 Symbolic computation of Gb in terms of exp(ψξ) and exp(-ψξ) completed.")

In [ ]:
# # Compute exponential times a cubic polynomial in ξ for the Green's function quadrature:
# # ∫ₐᵇ (c₀ + c₁ξ + c₂ξ² + c₃ξ³) * exp(α ξ) dξ
# c₀, c₁, c₂, c₃, α, a, b   = symbols("c₀ c₁ c₂ c₃ α a b",   real=true)
# integral_expr = (c₀ + c₁*ξ + c₂*ξ^2 + c₃*ξ^3) * exp(α * ξ)
# integral = integrate(integral_expr, (ξ, a, b))
# integral = simplify(integral)
# println("1.12 Symbolic computation of ∫ₐᵇ (c₀ + c₁ξ + c₂ξ² + c₃ξ³) * exp(α ξ) dξ completed.")
# display(simplify(integral.args[1].args[1]))

In [ ]:
# Compute $\partial_\xi G_b | \xi=0$ for Latex notes:
∂ξ_Gb = diff(Gb, ξ)
∂ξ_Gb_0 = ∂ξ_Gb.subs(ξ,0)
println("1.2 LaTeX vertical derivative wrt ξ of Gb at ξ=0:")
display(∂ξ_Gb_0)

In [ ]:
# Compute $z$ integral of Gb for Latex notes:
Gb_int = sympy.integrate(Gb,(z,z,0))
max_obj = sympy.Max(z, -H)
Gb_int = Gb_int.subs(max_obj, z)
Gb_int = sympy.simplify(Gb_int)
Latex_Gb_int_den = 2 * ψ^2 * κ * (exp(2 * ψ * H) + 1)
Latex_Gb_int_num = 
    exp(   ψ * ( z - ξ + 2 * H)) -
    exp(   ψ * ( z + ξ + 2 * H)) +
2 * exp(   ψ * ( ξ + 2 * H ) ) + 
2 * exp( - ψ * ξ) +  
    exp(   ψ * ( ξ - z )) - 
    exp( - ψ * ( z + ξ )) -
    exp(   ψ * ( ξ + 2 * H -  sympy.Max( z, ξ ))) - 
    exp( - ψ * ( ξ - 2 * H -  sympy.Max( z, ξ ) )) -
    exp(   ψ * ( ξ - sympy.Max( z, ξ ) )) - 
    exp( - ψ * ( ξ - sympy.Max( z, ξ ) )) 

@assert sympy.simplify(Latex_Gb_int_num / Latex_Gb_int_den - Gb_int) == 0
println("1.3 LaTeX Gb integral matches.")
display(Gb_int)

In [ ]:
# # Compute z integral of $\partial_\xi G_b | \xi=0$ for Latex notes:
# ∂ξ_Gb_int = simplify(sympy.integrate(∂ξ_Gb_0,(z, ξ, 0)))
# Latex_∂ξ_Gb_int_den = ψ * κ * (exp(2 * ψ * H(x,y)) + 1)
# Latex_∂ξ_Gb_int_num = - 1 - exp(ψ*(ξ + 2*H(x,y))) + exp(2 * ψ * H(x,y)) + exp(-ψ*ξ)
# @assert sympy.simplify(Latex_∂ξ_Gb_int_num / Latex_∂ξ_Gb_int_den - ∂ξ_Gb_int) == 0
# println("1.31 LaTeX ∂ξ Gb | ξ=0 integral matches.")
# display(∂ξ_Gb_int)

In [ ]:
# Compute z integral of $\partial_\xi G_b | \xi=0$ for Latex notes:
∂ξ_Gb_int = simplify(sympy.integrate(∂ξ_Gb_0,(z, ξ, 0)))
Latex_∂ξ_Gb_int_den = ψ * κ * (exp(2 * ψ * H) + 1)
Latex_∂ξ_Gb_int_num = - 1 - exp(ψ*(ξ + 2*H)) + exp(2 * ψ * H) + exp(-ψ*ξ)
@assert sympy.simplify(Latex_∂ξ_Gb_int_num / Latex_∂ξ_Gb_int_den - ∂ξ_Gb_int) == 0
println("1.31 LaTeX ∂ξ Gb | ξ=0 integral matches.")
display(∂ξ_Gb_int)

In [ ]:
# Check that solution satisfies the ODE:
b = - (Γ/Ek) * integrate(Gb * B(x,y,ξ), (ξ, -H, 0)) + ∂ξ_Gb_0 * bs(x,y)
b = simplify(b.subs(max_obj, z).subs(Γ,Γ_subs))
ode_check = sympy.simplify(Ek * diff(κ * diff(b,z),z) - Γ_subs * (b - B(x,y,z)))
@assert ode_check == 0
println("1.4 ODE check passed.")

In [ ]:
# Check integration over cubic polynomial for the cardinal B-spline function
# tmp_int = simplify(Gb * ξ^3)
# Gb_beta3_int = sympy.integrate(tmp_int, (ξ, -H, 0))
# Gb_beta3_int = Gb_beta3_int.subs(max_obj, z)
# Gb_beta3_int = simplify(Gb_beta3_int)
# Latex_Gb_beta3_int = -(z * (z^2 * ψ^2 + 6))/(κ * ψ^4) + 
# ((3 * ψ^2 * H^2 + 6) * exp(ψ * (z + H)) - (3 * ψ^2 * H^2 + 6) * exp(ψ * (H - z)) )/(κ * ψ^5 * (exp(2 * ψ * H) + 1))
# @assert Gb_beta3_int - Latex_Gb_beta3_int == 0
# println("1.5 Gb * ξ^3 LaTeX integral matches.")
# display(Latex_Gb_beta3_int)

### Make plots

In [ ]:
using CairoMakie
using QuadGK

CairoMakie.activate!()

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
H_val  = 1.0
Γ_val  = 1.0
κ_val = 1.0

# Choose Ekman numbers here
Ek1 = 0.25
Ek2 = 0.025

Ek_vals = [Ek1, Ek2]
ψ_vals  = [sqrt(Γ_val / (Ek * κ_val)) for Ek in Ek_vals]

zgrid = range(-H_val, 0.0, length=256)
ξ_vals = range(-H_val,0, length=5)
bs_vals = [0.0, 0.1]

# -------------------------------------------------------------------
# Publication-style theme
# -------------------------------------------------------------------
set_theme!(
    Theme(
        fontsize = 16,
        fonts = (; regular = "TeX Gyre Heros"),
        Axis = (
            titlesize = 20,
            xlabelsize = 18,
            ylabelsize = 18,
            xticklabelsize = 16,
            yticklabelsize = 16,
            xgridvisible = true,
            ygridvisible = true,
            xgridstyle = :dash,
            ygridstyle = :dash,
            xgridcolor = (:gray70, 0.35),
            ygridcolor = (:gray70, 0.35),
            spinewidth = 1.2,
            xticksize = 5,
            yticksize = 5,
            xminorticksvisible = false,
            yminorticksvisible = false,
            topspinevisible = true,
            rightspinevisible = true,
        ),
        Legend = (
            framevisible = false,
            labelsize = 15,
            titlesize = 16,
            patchsize = (22, 14),
        ),
        Lines = (
            linewidth = 2.5,
        )
    )
)

# -------------------------------------------------------------------
# Green's function G_b(z; ξ)
# -------------------------------------------------------------------
function G_b(z, ξ; H_val=1.0, ψ_val=1.0, κ_val=1.0)
    pref = exp(ψ_val * (z + ξ)) / (2 * κ_val * ψ_val * (exp(2ψ_val*H_val) + 1))
    if z <= ξ
        return pref * (1 - exp(-2ψ_val*ξ)) * (exp(2ψ_val*H_val) + exp(-2ψ_val*z))
    else
        return pref * (1 - exp(-2ψ_val*z)) * (exp(2ψ_val*H_val) + exp(-2ψ_val*ξ))
    end
end

# -------------------------------------------------------------------
# Analytic ∂_ξ G_b(z; 0)
# -------------------------------------------------------------------
function dGdξ_at0(z; H_val=1.0, ψ_val=1.0, κ_val=1.0)
    return exp(ψ_val*z) * (exp(2ψ_val*H_val) + exp(-2ψ_val*z)) / (κ_val * (exp(2ψ_val*H_val) + 1))
end

# -------------------------------------------------------------------
# Buoyancy profile b(z)
# -------------------------------------------------------------------
function b_profile(z; H_val=1.0, Γ_val=1.0, κ_val=1.0, ψ_val=sqrt(1/0.05), bs=0.0, Bfunc=ξ -> 0.0)
    Ek = Γ_val / (ψ_val^2 * κ_val)
    integrand(ξ) = G_b(z, ξ; H_val=H_val, ψ_val=ψ_val, κ_val=κ_val) * Bfunc(ξ)
    I, _ = quadgk(integrand, -H_val, 0.0)
    surf = dGdξ_at0(z; H_val=H_val, ψ_val=ψ_val, κ_val=κ_val) * bs
    return -(Γ_val/Ek) * I + surf
end

# -------------------------------------------------------------------
# Forcing / relaxation profiles
# -------------------------------------------------------------------
B2(ξ) = exp(4ξ / H_val) - 1
B3(ξ) = ξ

forcing_families = [
    ("Exponential forcing", B2),
    ("Linear forcing", B3)
]

colors = Dict(
    "Exponential forcing" => :royalblue,
    "Linear forcing"      => :darkorange
)

linestyles = Dict(
    0.0 => :solid,
    0.1 => :dash
)

# -------------------------------------------------------------------
# Precompute forcing profiles
# -------------------------------------------------------------------
B2_vals = [B2(z) for z in zgrid]
B3_vals = [B3(z) for z in zgrid]

# -------------------------------------------------------------------
# Compute buoyancy profiles for one Ek
# -------------------------------------------------------------------
function compute_profiles(Ek, ψ_val)
    profiles = Dict{Tuple{String, Float64}, Vector{Float64}}()
    for (fname, Bfunc) in forcing_families
        for bs in bs_vals
            profiles[(fname, bs)] = [
                b_profile(z; H_val=H_val, Γ_val=Γ_val, κ_val=κ_val, ψ_val=ψ_val, bs=bs, Bfunc=Bfunc)
                for z in zgrid
            ]
        end
    end
    return profiles
end

# -------------------------------------------------------------------
# Figure layout
# -------------------------------------------------------------------
fig = Figure(size = (1200, 1450))

Label(fig[0, 1:2],
    "Buoyancy solutions and Green's functions for two Ekman numbers",
    fontsize = 22,
    font = :bold
)

# -------------------------------------------------------------------
# Top row
# -------------------------------------------------------------------
profiles1 = compute_profiles(Ek1, ψ_vals[1])

ax11 = Axis(fig[1, 1],
    xlabel = "value",
    ylabel = "depth z/H",
    title  = "(a) Buoyancy solutions, Ek = $(Ek1)",
    aspect = 1.0
)

ax12 = Axis(fig[1, 2],
    xlabel = "value",
    ylabel = "",
    title  = "(b) Green's functions, Ek = $(Ek1)",
    aspect = 1.0
)

# Buoyancy panel
lines!(ax11, B2_vals, zgrid, color = colors["Exponential forcing"], linewidth = 4, alpha = 0.20)
lines!(ax11, B3_vals, zgrid, color = colors["Linear forcing"], linewidth = 4, alpha = 0.20)

for (fname, _) in forcing_families
    c = colors[fname]
    for bs in bs_vals
        vals = profiles1[(fname, bs)]
        lines!(ax11, vals, zgrid,
            color = c,
            linestyle = linestyles[bs],
            linewidth = 2.6
        )
    end
end

# Green's function panel
ξcols = cgrad(:viridis, length(ξ_vals), categorical = true)
for (j, ξ) in enumerate(ξ_vals)
    Gvals = [G_b(z, ξ; H_val=H_val, ψ_val=ψ_vals[1], κ_val=κ_val) for z in zgrid]
    lines!(ax12, Gvals, zgrid, color = ξcols[j], linewidth = 2.6)
end

# Derivative panel (top row)
dGvals1 = [dGdξ_at0(z; H_val=H_val, ψ_val=ψ_vals[1], κ_val=κ_val) for z in zgrid]
lines!(ax12, dGvals1.*Ek1, zgrid, color = :purple, linewidth = 2.8)

# -------------------------------------------------------------------
# Bottom row
# -------------------------------------------------------------------
profiles2 = compute_profiles(Ek2, ψ_vals[2])

ax21 = Axis(fig[2, 1],
    xlabel = "value",
    ylabel = "depth z/H",
    title  = "(c) Buoyancy solutions, Ek = $(Ek2)",
    aspect = 1.0
)

ax22 = Axis(fig[2, 2],
    xlabel = "value",
    ylabel = "",
    title  = "(d) Green's functions, Ek = $(Ek2)",
    aspect = 1.0
)

# Buoyancy panel
lines!(ax21, B2_vals, zgrid, color = colors["Exponential forcing"], linewidth = 4, alpha = 0.20)
lines!(ax21, B3_vals, zgrid, color = colors["Linear forcing"], linewidth = 4, alpha = 0.20)

for (fname, _) in forcing_families
    c = colors[fname]
    for bs in bs_vals
        vals = profiles2[(fname, bs)]
        lines!(ax21, vals, zgrid,
            color = c,
            linestyle = linestyles[bs],
            linewidth = 2.6
        )
    end
end

# Green's function panel
for (j, ξ) in enumerate(ξ_vals)
    Gvals = [G_b(z, ξ; H_val=H_val, ψ_val=ψ_vals[2], κ_val=κ_val) for z in zgrid]
    lines!(ax22, Gvals, zgrid, color = ξcols[j], linewidth = 2.6)
end

# Derivative panel (bottom row)
dGvals2 = [dGdξ_at0(z; H_val=H_val, ψ_val=ψ_vals[2], κ_val=κ_val) for z in zgrid]
lines!(ax22, dGvals2.*Ek2, zgrid, color = :purple, linewidth = 2.8)

# -------------------------------------------------------------------
# Axis linking and formatting
# -------------------------------------------------------------------
linkyaxes!(ax11, ax21)
linkyaxes!(ax12, ax22)

for ax in (ax11, ax12, ax21, ax22)
    ax.topspinevisible = true
    ax.rightspinevisible = true
end

# -------------------------------------------------------------------
# Legends below the panels
# -------------------------------------------------------------------
legend_b = Legend(fig[3, 1],
    [
        LineElement(color = colors["Exponential forcing"], linewidth = 3),
        LineElement(color = colors["Linear forcing"], linewidth = 3),
        LineElement(color = :black, linestyle = :solid, linewidth = 3),
        LineElement(color = :black, linestyle = :dash, linewidth = 3),
        # LineElement(color = colors["Exponential forcing"], linewidth = 4, alpha = 0.20),
    ],
    [
        "Exponential forcing family",
        "Linear forcing family",
        "b(z), b_s = 0.0",
        "b(z), b_s = 0.1",
        # "forcing profile B(z)"
    ],
    title = "Buoyancy panel",
    orientation = :horizontal,
    nbanks = 2,
    framevisible = false
)

legend_elements = LineElement[]
legend_labels = String[]

for (j, ξ) in enumerate(ξ_vals)
    push!(legend_elements, LineElement(color = ξcols[j], linewidth = 3))
    push!(legend_labels, "ξ/H = $(round(ξ/H_val, digits=2))")
end

push!(legend_elements, LineElement(color = :purple, linewidth = 3))
push!(legend_labels, "Ek ∂ξ G_b(z;0)")

legend_g = Legend(fig[3, 2],
    legend_elements,
    legend_labels,
    title = "Green's function panel",
    orientation = :horizontal,
    nbanks = 2,
    framevisible = false
)

# -------------------------------------------------------------------
# Tightened spacing
# -------------------------------------------------------------------
colgap!(fig.layout, 1, 10)
rowgap!(fig.layout, 1, 4)
rowgap!(fig.layout, 2, 2)

# -------------------------------------------------------------------
# Save to PDF
# -------------------------------------------------------------------
CairoMakie.save("buoyancy_solutions_and_Greens_functions.pdf", fig)

fig

In [ ]:
# -------------------------------------------------------------------
# Green's function heatmaps for the two Ek values
# -------------------------------------------------------------------

fig3 = Figure(size = (1400, 700))

# Compute both matrices first so we can use a common color scale

# Forcing-depth grid
xigrid = range(-H_val, 0.0, length=256)

Gmats = [
    [G_b(z, ξ; H_val=H_val, ψ_val=ψ_vals[1], κ_val=κ_val) for z in zgrid, ξ in xigrid],
    [G_b(z, ξ; H_val=H_val, ψ_val=ψ_vals[2], κ_val=κ_val) for z in zgrid, ξ in xigrid]
]

clim = maximum(abs, vcat(vec(Gmats[1]), vec(Gmats[2])))

# -------------------------------------------------------------------
# Left panel: Ek = Ek1
# -------------------------------------------------------------------
ax31 = Axis(fig3[1, 1],
    xlabel = "forcing depth ξ",
    ylabel = "response depth z/H",
    title  = "Green's function G_b(z; ξ), Ek = $(Ek1)",
    aspect = 1.0
)

hm1 = heatmap!(ax31, xigrid, zgrid, Gmats[1],
    colormap = :balance,
    colorrange = (-clim, clim)
)

contour!(ax31, xigrid, zgrid, Gmats[1],
    color = :black,
    linewidth = 0.7,
    levels = 12
)

ylims!(ax31, -H_val, 0.0)

# -------------------------------------------------------------------
# Right panel: Ek = Ek2
# -------------------------------------------------------------------
ax32 = Axis(fig3[1, 2],
    xlabel = "forcing depth ξ",
    ylabel = "",
    title  = "Green's function G_b(z; ξ), Ek = $(Ek2)",
    aspect = 1.0
)

hm2 = heatmap!(ax32, xigrid, zgrid, Gmats[2],
    colormap = :balance,
    colorrange = (-clim, clim)
)

contour!(ax32, xigrid, zgrid, Gmats[2],
    color = :black,
    linewidth = 0.7,
    levels = 12
)

ylims!(ax32, -H_val, 0.0)

# -------------------------------------------------------------------
# Shared colorbar
# -------------------------------------------------------------------
Colorbar(fig3[1, 3], hm1,
    label = "G_b",
    width = 18
)

# -------------------------------------------------------------------
# Cosmetics
# -------------------------------------------------------------------
for ax in (ax31, ax32)
    ax.topspinevisible = true
    ax.rightspinevisible = true
    ax.xgridvisible = true
    ax.ygridvisible = true
    ax.xgridstyle = :dash
    ax.ygridstyle = :dash
end

# Tight spacing
colgap!(fig3.layout, 1, 12)

# -------------------------------------------------------------------
# Save to PDF
# -------------------------------------------------------------------
CairoMakie.save("buoyancy_Greens_functions.pdf", fig3)

fig3